In [ ]:
"""
Stochastic Feature Selection & Energy Optimization: Boltzmann vs Cauchy Machines
--------------------------------------------------------------------------------
Dataset : Wine Quality (UCI 186 / Kaggle).  Target: quality >= 6  ->  good (1), else 0.

Run     : python wine_boltzmann_cauchy.py [winequality-red.csv [winequality-white.csv]]

Data lookup order
  1. file(s) given on the command line (';' or ',' separated, auto-detected)
  2. winequality-red.csv / winequality-white.csv / winequality.csv / WineQT.csv in the folder
  3. download of the UCI red-wine file
  4. synthetic wine-like data (so the program always runs; results are only a demo)

State vector s in {0,1}^n   (1 = feature selected)
Energy      E(s) = CV_error(s) + LAMBDA * (#selected / n)
  CV_error  = 3-fold misclassification rate of a ridge least-squares classifier
Boltzmann annealing : T_k = T0 / ln(1+k),   Gaussian-like local move (flip 1 bit)
Cauchy annealing    : T_k = T0 / (1+k),     Cauchy-distributed jump size (flip many bits sometimes)
Acceptance          : Metropolis, P = exp(-dE / T)

Requires: numpy, pandas
"""
import os
import sys
import itertools
import urllib.request

import numpy as np
import pandas as pd

SEED = 11
LAMBDA = 0.05          # weight of the feature-count term in the energy
T0 = 0.05              # initial temperature
UCI_RED = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
FEATURES = ["fixed acidity", "volatile acidity", "citric acid", "residual sugar", "chlorides",
            "free sulfur dioxide", "total sulfur dioxide", "density", "pH", "sulphates", "alcohol"]


# ----------------------------------------------------------------------------
# data
# ----------------------------------------------------------------------------
def _read(p):
    return pd.read_csv(p, sep=None, engine="python")


def load_data(paths):
    paths = [p for p in paths if p]
    if not paths:
        paths = [p for p in ("winequality-red.csv", "winequality-white.csv", "winequality.csv",
                             "WineQT.csv") if os.path.exists(p)]
    if paths:
        print("[data] Loading " + ", ".join(paths))
        df = pd.concat([_read(p) for p in paths], ignore_index=True)
        return _clean(df), "file: " + ", ".join(paths)
    try:
        print("[data] Trying to download UCI red wine data ...")
        urllib.request.urlretrieve(UCI_RED, "winequality-red.csv")
        return _clean(_read("winequality-red.csv")), "UCI download (red)"
    except Exception as e:
        print(f"[data] Download failed ({type(e).__name__}). Using SYNTHETIC wine-like data.")
        print("[data] Put winequality-red.csv (or WineQT.csv) next to this script for real data.")
        return synthetic(), "synthetic"


def _clean(df):
    df = df.copy()
    df.columns = [c.strip().lower().replace("_", " ") for c in df.columns]
    df = df.drop(columns=[c for c in ("id",) if c in df.columns])
    num = df.select_dtypes(include=[np.number]).dropna()
    if "quality" not in num.columns:
        raise ValueError("No 'quality' column found.")
    return num


def synthetic(n=1599):
    rng = np.random.default_rng(SEED)
    q = rng.choice([3, 4, 5, 6, 7, 8], n, p=[.01, .03, .43, .40, .12, .01])
    good = (q >= 6).astype(float)
    d = {
        "fixed acidity": rng.normal(8.3, 1.7, n), "volatile acidity": rng.normal(.65 - .15 * good, .15, n),
        "citric acid": rng.normal(.25 + .07 * good, .18, n), "residual sugar": rng.gamma(2, 1.2, n),
        "chlorides": rng.normal(.09, .03, n), "free sulfur dioxide": rng.gamma(3, 5, n),
        "total sulfur dioxide": rng.gamma(3, 15, n) - 8 * good, "density": rng.normal(.9967, .002, n),
        "pH": rng.normal(3.3, .15, n), "sulphates": rng.normal(.6 + .08 * good, .15, n),
        "alcohol": rng.normal(9.9 + 1.0 * good, 1.0, n), "quality": q}
    return pd.DataFrame(d)


# ----------------------------------------------------------------------------
# energy function
# ----------------------------------------------------------------------------
class Problem:
    def __init__(self, df, seed=SEED):
        self.names = [c for c in df.columns if c != "quality"]
        X = df[self.names].values.astype(float)
        y = (df["quality"].values >= 6).astype(float)
        rng = np.random.default_rng(seed)
        idx = rng.permutation(len(y))
        nt = int(0.8 * len(y))
        self.tr, self.te = idx[:nt], idx[nt:]
        mu, sd = X[self.tr].mean(0), X[self.tr].std(0) + 1e-9
        self.X = (X - mu) / sd
        self.y = y
        self.n = X.shape[1]
        self.folds = np.array_split(self.tr, 3)
        self.cache = {}
        self.evals = 0                       # number of UNIQUE energy evaluations
        self.calls = 0

    @staticmethod
    def key(s): return tuple(int(v) for v in s)

    def fit_predict(self, cols, tr, te):
        if len(cols) == 0:
            return np.full(len(te), float(self.y[tr].mean() >= 0.5))
        A = np.hstack([self.X[np.ix_(tr, cols)], np.ones((len(tr), 1))])
        B = np.hstack([self.X[np.ix_(te, cols)], np.ones((len(te), 1))])
        w = np.linalg.solve(A.T @ A + 1e-3 * np.eye(A.shape[1]), A.T @ self.y[tr])
        return (B @ w >= 0.5).astype(float)

    def cv_error(self, s):
        cols = list(np.where(np.asarray(s) == 1)[0])
        errs = []
        for f in range(3):
            te = self.folds[f]
            tr = np.concatenate([self.folds[g] for g in range(3) if g != f])
            errs.append(np.mean(self.fit_predict(cols, tr, te) != self.y[te]))
        return float(np.mean(errs))

    def test_error(self, s):
        cols = list(np.where(np.asarray(s) == 1)[0])
        return float(np.mean(self.fit_predict(cols, self.tr, self.te) != self.y[self.te]))

    def energy(self, s):
        k = self.key(s)
        self.calls += 1
        if k not in self.cache:
            self.evals += 1
            self.cache[k] = self.cv_error(s) + LAMBDA * (sum(k) / self.n)
        return self.cache[k]

    def parts(self, s):
        return self.cv_error(s), sum(self.key(s))

    def names_of(self, s):
        return [self.names[i] for i in range(self.n) if s[i]]


# ----------------------------------------------------------------------------
# annealing machines
# ----------------------------------------------------------------------------
def temp(kind, k):
    return T0 / np.log(1.0 + k + 1.0) if kind == "boltzmann" else T0 / (1.0 + k)


def propose(kind, s, T, rng):
    n = len(s); new = s.copy()
    if kind == "boltzmann":
        m = 1                                              # local move
    else:                                                  # Cauchy jump: heavy-tailed size
        c = abs(np.tan(np.pi * (rng.random() - 0.5)))
        m = int(np.clip(1 + int(c * (0.5 + 2 * T / T0)), 1, n))
    for i in rng.choice(n, m, replace=False):
        new[i] ^= 1
    return new, m


def anneal(prob, kind, s0, iters=300, seed=0, T_fixed=None, log=False):
    """Returns dict(best_s, best_E, trace, log, evals_to_best, accepted, uphill)."""
    rng = np.random.default_rng(seed)
    s = np.array(s0, dtype=int); E = prob.energy(s)
    best_s, best_E = s.copy(), E
    trace = [E]; events = []; acc = up = 0; first_best = 0
    ev0 = len(prob.cache)
    for k in range(iters):
        T = T_fixed if T_fixed is not None else temp(kind, k)
        new, m = propose(kind, s, T, rng)
        En = prob.energy(new); dE = En - E
        if dE <= 0 or rng.random() < np.exp(-dE / max(T, 1e-12)):
            if dE > 0: up += 1
            acc += 1
            if log: events.append((k, T, m, dE, True))
            s, E = new, En
            if E < best_E - 1e-12:
                best_s, best_E, first_best = s.copy(), E, k + 1
        elif log:
            events.append((k, T, m, dE, False))
        trace.append(E)
    return dict(best_s=best_s, best_E=best_E, trace=np.array(trace), events=events,
                iters_to_best=first_best, accepted=acc, uphill=up, final_s=s, final_E=E)


def greedy_descent(prob, s):
    s = np.array(s); E = prob.energy(s)
    while True:
        best = None
        for i in range(len(s)):
            t = s.copy(); t[i] ^= 1; e = prob.energy(t)
            if e < E - 1e-12 and (best is None or e < best[0]): best = (e, t)
        if best is None: return s, E
        E, s = best


def spark(v, width=60):
    v = np.asarray(v); chars = "▁▂▃▄▅▆▇█"
    idx = np.linspace(0, len(v) - 1, min(width, len(v))).astype(int)
    w = v[idx]; lo, hi = w.min(), w.max()
    return "".join(chars[int((x - lo) / (hi - lo + 1e-12) * (len(chars) - 1))] for x in w)


def fmt(s): return "".join(str(int(v)) for v in s)


# ----------------------------------------------------------------------------
# application
# ----------------------------------------------------------------------------
def ask_int(msg, lo, hi, default):
    s = input(f"{msg} [{lo}-{hi}] (default {default}): ").strip()
    try:
        v = int(s)
        if lo <= v <= hi: return v
    except ValueError:
        pass
    return default


class App:
    def __init__(self, df, source):
        self.P = Problem(df); self.source = source
        self.rng = np.random.default_rng(SEED)
        self.s0 = None
        self.results = {"boltzmann": [], "cauchy": []}      # all finished runs
        self.best = None                                    # global best (s, E)
        self.exact = None
        self.make_state("random", silent=True)

    def note(self, s, E, who):
        if self.best is None or E < self.best[1] - 1e-12:
            self.best = (np.array(s), E, who)

    def show_state(self, s, label="State"):
        P = self.P
        err, k = P.parts(s)
        print(f"{label}: {fmt(s)}   selected {k}/{P.n}   CV error {err*100:.2f}%   Energy {P.energy(s):.4f}")
        print("   features: " + (", ".join(P.names_of(s)) or "(none)"))

    def make_state(self, mode, silent=False):
        n = self.P.n
        if mode == "random": s = self.rng.integers(0, 2, n)
        elif mode == "all": s = np.ones(n, int)
        elif mode == "none": s = np.zeros(n, int)
        else: s = np.array([int(c) for c in mode])
        self.s0 = s
        if not silent: self.show_state(s, "Initial state")

    # -- 1
    def op1(self):
        print("1) random   2) all features   3) no features   4) enter bit string")
        c = input("Choice (default 1): ").strip() or "1"
        if c == "2": self.make_state("all")
        elif c == "3": self.make_state("none")
        elif c == "4":
            b = input(f"Enter {self.P.n} bits: ").strip()
            if len(b) == self.P.n and set(b) <= {"0", "1"}: self.make_state(b)
            else: print("Invalid, keeping previous."); self.show_state(self.s0, "Initial state")
        else: self.make_state("random")
        print("\nFeature index map:")
        for i, n in enumerate(self.P.names): print(f"  bit {i:>2}: {n}")

    def run(self, kind, iters, seed=None, s0=None, **kw):
        seed = int(self.rng.integers(1 << 30)) if seed is None else seed
        r = anneal(self.P, kind, self.s0 if s0 is None else s0, iters, seed, **kw)
        self.note(r["best_s"], r["best_E"], kind)
        return r

    def machine(self, kind):
        iters = ask_int("Annealing iterations", 20, 5000, 300)
        self.show_state(self.s0, "Start")
        r = self.run(kind, iters)
        self.results[kind].append(r)
        print(f"\n{kind.capitalize()} machine finished: {r['accepted']} accepted moves "
              f"({r['uphill']} uphill), best found after {r['iters_to_best']} iterations, "
              f"{len(self.P.cache)} distinct states evaluated")
        print("Energy trace: " + spark(r["trace"]))
        print(f"   T start {temp(kind,0):.4f} -> T end {temp(kind, iters-1):.5f}")
        self.show_state(r["best_s"], "Best subset")
        print(f"   held-out TEST error of best subset: {self.P.test_error(r['best_s'])*100:.2f}%  "
              f"(all features: {self.P.test_error(np.ones(self.P.n, int))*100:.2f}%)")

    def op2(self): self.machine("boltzmann")
    def op3(self): self.machine("cauchy")

    # -- 4
    def op4(self):
        k = input("Machine (b = Boltzmann, c = Cauchy; default b): ").strip().lower()
        kind = "cauchy" if k.startswith("c") else "boltzmann"
        iters = ask_int("Iterations", 20, 2000, 200)
        r = self.run(kind, iters, log=True)
        ev = r["events"]
        down = [e for e in ev if e[4] and e[3] <= 0]; upm = [e for e in ev if e[4] and e[3] > 0]
        rej = [e for e in ev if not e[4]]
        print(f"\n{kind.capitalize()} transitions over {iters} proposals:")
        print(f"  downhill accepted : {len(down)}\n  uphill accepted   : {len(upm)}\n  rejected          : {len(rej)}")
        if upm: print(f"  mean uphill dE accepted: {np.mean([e[3] for e in upm]):+.4f}")
        if rej: print(f"  mean dE rejected       : {np.mean([e[3] for e in rej]):+.4f}")
        mv = [e[2] for e in ev]
        print(f"  bits flipped per move: mean {np.mean(mv):.2f}, max {max(mv)}")
        print("\nPhase breakdown (quarters of the run):")
        q = max(1, iters // 4)
        print(f"{'Phase':>6}{'T':>10}{'accept %':>10}{'uphill':>8}{'E end':>9}")
        for p in range(4):
            seg = ev[p * q:(p + 1) * q]
            if not seg: continue
            print(f"{p+1:>6}{seg[0][1]:>10.5f}{np.mean([e[4] for e in seg])*100:>9.1f}%"
                  f"{sum(1 for e in seg if e[4] and e[3] > 0):>8}{r['trace'][min(iters,(p+1)*q)]:>9.4f}")
        print("\nFirst 12 transitions:")
        print(f"{'it':>4}{'T':>10}{'flips':>6}{'dE':>10}  result")
        for e in ev[:12]:
            print(f"{e[0]:>4}{e[1]:>10.5f}{e[2]:>6}{e[3]:>+10.4f}  {'accepted' if e[4] else 'rejected'}")
        print("\nEnergy trace: " + spark(r["trace"]))
        self.show_state(r["best_s"], "Best")
        self.results[kind].append(r)

    # -- 5
    def op5(self):
        P = self.P
        trials = ask_int("Escape trials per machine", 5, 200, 40)
        s_loc, E_loc = greedy_descent(P, self.s0)
        self.show_state(s_loc, "Local minimum reached by greedy bit-flip descent")
        # is it a true local min, and is it the global one?
        glob = self.exhaustive()
        gE = glob[0][0]
        print(f"Global minimum energy (exhaustive search): {gE:.4f}   "
              f"{'-> this IS the global minimum' if E_loc <= gE + 1e-9 else '-> this is a TRAP (higher than global)'}")
        T_fix = float(input(f"Fixed escape temperature (default {T0/2:.3f}): ") or T0 / 2)
        print(f"\nRelease from the minimum at fixed T={T_fix}, 120 steps per trial, {trials} trials")
        print(f"{'Machine':<11}{'escaped':>9}{'found better':>14}{'reached global':>16}{'mean best E':>13}")
        for kind in ("boltzmann", "cauchy"):
            esc = bet = gl = 0; bs = []
            for t in range(trials):
                r = anneal(P, kind, s_loc, 120, seed=1000 + t, T_fixed=T_fix)
                left = any(not np.array_equal(P.key(x), P.key(s_loc)) for x in [r["final_s"], r["best_s"]]) \
                    or r["uphill"] > 0
                esc += left; bet += r["best_E"] < E_loc - 1e-9; gl += r["best_E"] <= gE + 1e-9
                bs.append(r["best_E"]); self.note(r["best_s"], r["best_E"], kind)
            print(f"{kind:<11}{esc/trials*100:>8.0f}%{bet/trials*100:>13.0f}%{gl/trials*100:>15.0f}%{np.mean(bs):>13.4f}")
        print("\n'escaped' = accepted at least one uphill move or ended elsewhere.")
        print("Cauchy jumps can leap over barriers that single-bit Boltzmann moves must climb step by step.")

    # -- 6
    def op6(self):
        runs = ask_int("Independent runs per machine", 3, 100, 20)
        iters = ask_int("Iterations per run", 50, 2000, 250)
        P = self.P
        for kind in ("boltzmann", "cauchy"):
            sols = []
            for r in range(runs):
                s0 = self.rng.integers(0, 2, P.n)
                res = self.run(kind, iters, s0=s0); sols.append(res["best_s"])
                self.results[kind].append(res)
            S = np.array(sols); freq = S.mean(0)
            jac = [(np.sum(a & b) / max(1, np.sum(a | b))) for a, b in itertools.combinations(S, 2)]
            uniq = len({P.key(s) for s in S})
            print(f"\n=== {kind.upper()} : {runs} runs ===")
            print(f"{'Feature':<22}{'selected in':>12}  stability")
            for i in np.argsort(-freq):
                tag = "STABLE" if freq[i] >= .8 else ("never" if freq[i] <= .1 else "unstable")
                print(f"{P.names[i]:<22}{freq[i]*100:>10.0f}%  {'#'*int(freq[i]*20):<20} {tag}")
            print(f"Distinct final subsets: {uniq}/{runs}   mean pairwise Jaccard: {np.mean(jac):.3f}   "
                  f"mean #features: {S.sum(1).mean():.2f}")

    # -- exhaustive reference (2^n states, cached)
    def exhaustive(self):
        if self.exact is None:
            allv = [(self.P.energy(np.array(b)), b) for b in itertools.product((0, 1), repeat=self.P.n)]
            allv.sort(key=lambda t: t[0]); self.exact = allv
            self.note(np.array(allv[0][1]), allv[0][0], "exhaustive")
        return self.exact

    # -- 7
    def op7(self):
        runs = ask_int("Runs per machine", 5, 100, 30)
        iters = ask_int("Iterations per run", 50, 2000, 200)
        P = self.P
        gE = self.exhaustive()[0][0]
        res = {}
        for kind in ("boltzmann", "cauchy"):
            rs = []
            for r in range(runs):
                s0 = np.random.default_rng(500 + r).integers(0, 2, P.n)      # same starts for both
                rs.append(anneal(P, kind, s0, iters, seed=900 + r))
                self.note(rs[-1]["best_s"], rs[-1]["best_E"], kind)
            res[kind] = rs; self.results[kind] += rs
        print(f"\nGlobal optimum energy (exhaustive): {gE:.4f}\n")
        print(f"{'Metric':<34}{'Boltzmann':>12}{'Cauchy':>12}")
        def row(name, f, p=4):
            print(f"{name:<34}{f(res['boltzmann']):>12.{p}f}{f(res['cauchy']):>12.{p}f}")
        row("Mean best energy", lambda R: np.mean([r['best_E'] for r in R]))
        row("Std of best energy", lambda R: np.std([r['best_E'] for r in R]))
        row("Worst best energy", lambda R: np.max([r['best_E'] for r in R]))
        row("Runs reaching global optimum (%)", lambda R: np.mean([r['best_E'] <= gE + 1e-9 for r in R]) * 100, 1)
        row("Mean iterations to best", lambda R: np.mean([r['iters_to_best'] for r in R]), 1)
        row("Mean uphill moves accepted", lambda R: np.mean([r['uphill'] for r in R]), 1)
        row("Mean #features in best", lambda R: np.mean([r['best_s'].sum() for r in R]), 2)
        wins = sum(a['best_E'] < b['best_E'] - 1e-12 for a, b in zip(res['boltzmann'], res['cauchy']))
        losses = sum(b['best_E'] < a['best_E'] - 1e-12 for a, b in zip(res['boltzmann'], res['cauchy']))
        print(f"\nHead-to-head (same start): Boltzmann wins {wins}, Cauchy wins {losses}, ties {runs-wins-losses}")
        mb = np.mean([r['best_E'] for r in res['boltzmann']]); mc = np.mean([r['best_E'] for r in res['cauchy']])
        print("WINNER: " + ("Boltzmann" if mb < mc - 1e-9 else "Cauchy" if mc < mb - 1e-9 else "TIE") +
              " (lower mean best energy)")

    # -- 8
    def op8(self):
        P = self.P
        ex = self.exhaustive()
        print(f"Exhaustive search over all 2^{P.n} = {2**P.n} feature subsets\n")
        print("Top 5 subsets by energy:")
        print(f"{'Rank':>4}{'State':>14}{'k':>4}{'CV err':>9}{'Energy':>9}  features")
        for r, (E, b) in enumerate(ex[:5], 1):
            err, k = P.parts(np.array(b))
            print(f"{r:>4}{fmt(b):>14}{k:>4}{err*100:>8.2f}%{E:>9.4f}  {', '.join(P.names_of(b))}")
        E, b = ex[0]; b = np.array(b)
        full = np.ones(P.n, int)
        print(f"\nBEST SUBSET: {fmt(b)}  ({int(b.sum())} of {P.n} features)")
        print(f"  Held-out TEST error : {P.test_error(b)*100:.2f}%   (all {P.n} features: {P.test_error(full)*100:.2f}%)")
        print(f"  Energy              : {E:.4f}      (all features: {P.energy(full):.4f})")
        if self.best:
            s, e, who = self.best
            print(f"\nBest found by annealing so far: {fmt(s)}  E={e:.4f}  ({who})")

    # -- 9
    def op9(self):
        P = self.P; ex = self.exhaustive(); gE, gb = ex[0]
        print("=" * 64); print("STOCHASTIC OPTIMIZATION REPORT"); print("=" * 64)
        print(f"Dataset       : {self.source}  ({len(P.y)} samples, {P.n} features)")
        print(f"Task          : good wine (quality>=6) vs not;  ridge classifier, 3-fold CV")
        print(f"Energy        : E = CV_error + {LAMBDA} * (k/n);  search space 2^{P.n} = {2**P.n}")
        print(f"Schedules     : Boltzmann T=T0/ln(1+k), Cauchy T=T0/(1+k), T0={T0}")
        print(f"Global optimum: {fmt(gb)}  E={gE:.4f}  ({', '.join(P.names_of(gb))})")
        print(f"Baselines     : all features E={P.energy(np.ones(P.n,int)):.4f}   "
              f"no features E={P.energy(np.zeros(P.n,int)):.4f}")
        for kind in ("boltzmann", "cauchy"):
            R = self.results[kind]
            if not R: print(f"\n{kind.capitalize()}: no runs yet (use menu options 2-7)."); continue
            be = np.array([r['best_E'] for r in R])
            print(f"\n{kind.capitalize()}: {len(R)} runs   best E {be.min():.4f}   mean {be.mean():.4f}   "
                  f"reached optimum {np.mean(be <= gE + 1e-9)*100:.0f}%   "
                  f"mean #features {np.mean([r['best_s'].sum() for r in R]):.1f}")
        print(f"\nDistinct states evaluated: {len(P.cache)} of {2**P.n}")
        print(f"Test error of global-optimum subset: {P.test_error(np.array(gb))*100:.2f}% "
              f"vs all features {P.test_error(np.ones(P.n,int))*100:.2f}%")


def main():
    # We only treat arguments ending in ".csv" as explicit input paths
    args = [arg for arg in sys.argv[1:] if arg.endswith('.csv')]
    df, source = load_data(args)
    print(f"[data] {len(df)} wines, {df.shape[1]-1} features.\n")
    app = App(df, source)
    print("Initial state (random):"); app.show_state(app.s0)
    ops = {"1": app.op1, "2": app.op2, "3": app.op3, "4": app.op4, "5": app.op5,
           "6": app.op6, "7": app.op7, "8": app.op8, "9": app.op9}
    while True:
        print("\n===== BOLTZMANN / CAUCHY FEATURE SELECTION =====")
        print("1. Generate Initial Feature State")
        print("2. Boltzmann Machine Feature Selection")
        print("3. Cauchy Machine Feature Selection")
        print("4. Energy State Transition Analyzer")
        print("5. Local Minimum Escape Challenge")
        print("6. Feature Stability Analyzer")
        print("7. Boltzmann vs Cauchy Optimization Battle")
        print("8. Best Feature Subset Finder")
        print("9. Stochastic Optimization Report")
        print("10. Exit")
        try:
            c = input("Choice: ").strip()
        except EOFError:
            break
        if c == "10": print("Goodbye."); break
        if c in ops: print(); ops[c]()
        else: print("Invalid choice.")


if __name__ == "__main__":
    main()

[data] Trying to download UCI red wine data ...
[data] 1599 wines, 11 features.

Initial state (random):
State: 00101110000   selected 4/11   CV error 36.67%   Energy 0.3849
   features: citric acid, chlorides, free sulfur dioxide, total sulfur dioxide

===== BOLTZMANN / CAUCHY FEATURE SELECTION =====
1. Generate Initial Feature State
2. Boltzmann Machine Feature Selection
3. Cauchy Machine Feature Selection
4. Energy State Transition Analyzer
5. Local Minimum Escape Challenge
6. Feature Stability Analyzer
7. Boltzmann vs Cauchy Optimization Battle
8. Best Feature Subset Finder
9. Stochastic Optimization Report
10. Exit
Choice: 1

1) random   2) all features   3) no features   4) enter bit string
Choice (default 1): 2
Initial state: 11111111111   selected 11/11   CV error 26.12%   Energy 0.3112
   features: fixed acidity, volatile acidity, citric acid, residual sugar, chlorides, free sulfur dioxide, total sulfur dioxide, density, ph, sulphates, alcohol

Feature index map:
  bit  0: fix

In [ ]:
"""
Exercise 8 - Self-Organizing Map (SOM) Customer Segmentation and Cluster Exploration
Dataset : Mall Customers (Age, Annual Income (k$), Spending Score (1-100))

* Features are min-max normalised before training.
* BMU (Best Matching Unit) = neuron with the smallest Euclidean distance to the input.
* SOM : the BMU *and its neighbours* on a 1-D neuron chain move toward the input
        (Gaussian neighbourhood, learning rate and radius decay over time).
* CLN : Competitive Learning Network = winner-take-all; only the BMU moves.
Each neuron is one cluster; cluster k = neuron k.

Usage:  python som_customer_segmentation.py [path/to/Mall_Customers.csv]
If no CSV is found a SYNTHETIC stand-in with the same columns is generated (flagged).
"""
import math
import os
import sys

import numpy as np

SEED = 42
FEATS = ["Age", "Annual Income (k$)", "Spending Score"]
DEF_LR, DEF_ITERS, DEF_N = 0.5, 3000, 4


# ----------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------
def make_synthetic_mall(seed=1):
    rng = np.random.default_rng(seed)
    # five blobs mimicking the structure of the real Mall Customers data
    spec = [  # n, age(mu,sd), income(mu,sd), spending(mu,sd)
        (39, (25, 5), (26, 7), (78, 10)),
        (23, (45, 11), (26, 7), (21, 10)),
        (80, (43, 14), (55, 8), (50, 8)),
        (35, (41, 11), (88, 16), (17, 10)),
        (23, (32, 4), (86, 16), (82, 8)),
    ]
    rows = []
    for n, a, i, s in spec:
        rows.append(np.column_stack([rng.normal(*a, n), rng.normal(*i, n), rng.normal(*s, n)]))
    X = np.vstack(rows)
    X[:, 0] = np.clip(np.rint(X[:, 0]), 18, 70)
    X[:, 1] = np.clip(np.rint(X[:, 1]), 15, 137)
    X[:, 2] = np.clip(np.rint(X[:, 2]), 1, 99)
    X = X[rng.permutation(len(X))]
    ids = np.arange(1, len(X) + 1)
    return ids, X


def load_mall(path=None):
    cands = ([path] if path else []) + ["Mall_Customers.csv", "mall_customers.csv", "Mall Customers.csv"]
    for p in cands:
        if p and os.path.exists(p):
            import csv
            with open(p, newline="") as f:
                rows = list(csv.reader(f))
            head = [h.strip() for h in rows[0]]
            def col(*names):
                for n in names:
                    for i, h in enumerate(head):
                        if h.lower() == n.lower():
                            return i
                raise KeyError(names)
            i_id = col("CustomerID", "Customer ID")
            i_age = col("Age")
            i_inc = col("Annual Income (k$)", "Annual_Income_(k$)", "Annual Income")
            i_sp = col("Spending Score (1-100)", "Spending_Score", "Spending Score")
            ids = np.array([int(r[i_id]) for r in rows[1:] if r])
            X = np.array([[float(r[i_age]), float(r[i_inc]), float(r[i_sp])] for r in rows[1:] if r])
            return ids, X, p
    print("\n" + "!" * 78)
    print("! WARNING: Mall_Customers.csv not found -> using SYNTHETIC DEMO DATA (200 customers)")
    print("! Put Mall_Customers.csv next to this script (or pass its path) for the real data.")
    print("! Results below are NOT real mall-customer results.")
    print("!" * 78)
    ids, X = make_synthetic_mall()
    return ids, X, "SYNTHETIC"


# ----------------------------------------------------------------------------
# SOM / Competitive learning network
# ----------------------------------------------------------------------------
class SOM:
    def __init__(self, n, lr=DEF_LR, iters=DEF_ITERS, competitive=False, seed=SEED):
        self.n, self.lr, self.iters, self.competitive, self.seed = n, lr, iters, competitive, seed
        self.W = None

    def fit(self, X):
        rng = np.random.default_rng(self.seed)
        self.W = X[rng.choice(len(X), self.n, replace=False)].astype(float).copy()
        sigma0 = max(self.n / 2.0, 1.0)
        pos = np.arange(self.n)
        for t in range(self.iters):
            frac = t / self.iters
            lr_t = self.lr * math.exp(-3.0 * frac)
            x = X[rng.integers(len(X))]
            b = int(np.argmin(np.linalg.norm(self.W - x, axis=1)))       # BMU (Euclidean)
            if self.competitive:
                self.W[b] += lr_t * (x - self.W[b])                      # winner-take-all update
            else:
                sigma = max(sigma0 * math.exp(-3.0 * frac), 0.3)
                h = np.exp(-((pos - b) ** 2) / (2 * sigma ** 2))         # neighbourhood function
                self.W += lr_t * h[:, None] * (x - self.W)
        return self

    def distances(self, x):
        return np.linalg.norm(self.W - x, axis=1)

    def bmu(self, X):
        D = np.linalg.norm(X[:, None, :] - self.W[None, :, :], axis=2)
        idx = D.argmin(axis=1)
        return idx, D[np.arange(len(X)), idx]

    def quantization_error(self, X):
        return float(self.bmu(X)[1].mean())


def hr(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)


def ask(prompt, default, cast=int, lo=None, hi=None):
    try:
        s = input(f"{prompt} [{default}]: ").strip()
    except EOFError:
        return default
    try:
        v = cast(s) if s else default
    except ValueError:
        return default
    if lo is not None:
        v = max(lo, v)
    if hi is not None:
        v = min(hi, v)
    return v


class System:
    def __init__(self, path=None):
        self.ids, self.raw, src = load_mall(path)
        self.lo, self.hi = self.raw.min(0), self.raw.max(0)
        self.X = self.norm(self.raw)
        self.p33, self.p67 = np.percentile(self.raw, 33.3, axis=0), np.percentile(self.raw, 66.7, axis=0)
        self.som = None
        print(f"[data] source = {src} | {len(self.ids)} customers | features: {', '.join(FEATS)} "
              f"(min-max normalised to [0,1])")

    def norm(self, a):
        return (a - self.lo) / (self.hi - self.lo + 1e-12)

    def denorm(self, a):
        return a * (self.hi - self.lo) + self.lo

    def train(self, n, lr=DEF_LR, iters=DEF_ITERS, competitive=False):
        return SOM(n, lr, iters, competitive).fit(self.X)

    def ensure(self):
        if self.som is None:
            print(f"[info] No SOM trained yet -> training a default SOM with {DEF_N} neurons.")
            self.som = self.train(DEF_N)
        return self.som

    def describe(self, proto):
        age, inc, sp = proto
        a = "Young" if age <= self.p33[0] else "Older" if age >= self.p67[0] else "Middle-Aged"
        i = "Low" if inc <= self.p33[1] else "High" if inc >= self.p67[1] else "Moderate"
        s = "Low" if sp <= self.p33[2] else "High" if sp >= self.p67[2] else "Average"
        return f"{a}, {i}-Income Customers with {s} Spending"

    def cluster_sizes(self, som):
        idx, _ = som.bmu(self.X)
        return np.bincount(idx, minlength=som.n), idx

    # --- 1 -----------------------------------------------------------------
    def m1(self):
        hr("1. CUSTOMER SEGMENTATION USING SOM")
        n = ask("Number of SOM neurons", DEF_N, int, 2, 30)
        lr = ask("Initial learning rate", DEF_LR, float, 0.01, 1.0)
        it = ask("Training iterations", DEF_ITERS, int, 100, 100000)
        self.som = self.train(n, lr, it)
        sizes, idx = self.cluster_sizes(self.som)
        print(f"\nTrained SOM: {n} neurons, lr = {lr}, iterations = {it}, "
              f"quantization error = {self.som.quantization_error(self.X):.4f}")
        for k in range(n):
            members = np.where(idx == k)[0]
            proto = self.denorm(self.som.W[k])
            print(f"\nCluster {k+1} (neuron {k+1}) - {len(members)} customers"
                  f"  | prototype Age {proto[0]:.1f}, Income {proto[1]:.1f}k$, Spending {proto[2]:.1f}")
            if not len(members):
                print("   (empty cluster - no customer won this neuron)")
                continue
            print(f"   {'CustID':>6} {'Age':>5} {'Income(k$)':>11} {'Spending':>9}")
            for j in members:
                a, i_, s = self.raw[j]
                print(f"   {self.ids[j]:>6} {a:>5.0f} {i_:>11.0f} {s:>9.0f}")
        print("\nSummary:  " + "   ".join(f"C{k+1}: {sizes[k]}" for k in range(n)))

    # --- 2 -----------------------------------------------------------------
    def find_customer(self):
        cid = ask(f"Customer ID ({self.ids.min()}-{self.ids.max()})", int(self.ids[0]), int)
        w = np.where(self.ids == cid)[0]
        if not len(w):
            print(f"Customer {cid} not found -> using customer {self.ids[0]}.")
            return 0
        return int(w[0])

    def m2(self):
        hr("2. BEST MATCHING UNIT EXPLORER")
        som = self.ensure()
        j = self.find_customer()
        x = self.X[j]
        d = som.distances(x)
        b = int(d.argmin())
        a, i_, s = self.raw[j]
        print(f"\nCustomer {self.ids[j]}: Age {a:.0f}, Annual Income {i_:.0f}k$, Spending Score {s:.0f}")
        print(f"Normalised input vector: [{x[0]:.3f}, {x[1]:.3f}, {x[2]:.3f}]\n")
        print(f"{'Neuron':>6} {'Weight vector (normalised)':>30} {'Euclidean dist':>15}")
        for k in range(som.n):
            w = som.W[k]
            print(f"{k+1:>6} {'['+', '.join(f'{v:.3f}' for v in w)+']':>30} {d[k]:>15.4f}"
                  f"{'   <-- BMU' if k == b else ''}")
        proto = self.denorm(som.W[b])
        print(f"\nBest Matching Unit : neuron {b+1} (distance {d[b]:.4f})")
        print(f"Winning cluster    : Cluster {b+1} - {self.describe(proto)}")

    # --- 3 -----------------------------------------------------------------
    def imbalance(self, sizes):
        ideal = len(self.ids) / len(sizes)
        big = [k for k, s in enumerate(sizes) if s > 1.5 * ideal]
        small = [k for k, s in enumerate(sizes) if 0 < s < 0.5 * ideal]
        return big, small

    def m3(self):
        hr("3. DYNAMIC CLUSTER COUNT CHALLENGE")
        print("Rules: empty = 0 customers; imbalanced = cluster larger than 1.5x or smaller than "
              "0.5x the ideal share (N/neurons).\n")
        res = []
        for n in (2, 3, 4, 5, 6):
            som = self.train(n)
            sizes, _ = self.cluster_sizes(som)
            empty = [k for k in range(n) if sizes[k] == 0]
            big, small = self.imbalance(sizes)
            qe = som.quantization_error(self.X)
            res.append(dict(n=n, sizes=sizes, empty=empty, big=big, small=small, qe=qe))
            print(f"{n} neurons | sizes: " + ", ".join(f"C{k+1}={sizes[k]}" for k in range(n)))
            print(f"           empty clusters: {[k+1 for k in empty] or 'none'} | "
                  f"too large: {[k+1 for k in big] or 'none'} | too small: {[k+1 for k in small] or 'none'} | "
                  f"QE = {qe:.4f}")
        valid = [r for r in res if not r["empty"] and not r["big"] and not r["small"]]
        pool = valid or [r for r in res if not r["empty"]] or res
        pick, reason = pool[-1], "no elbow found, so the balanced configuration with the lowest QE"
        for a, b in zip(pool, pool[1:]):
            if b["n"] - a["n"] == 1 and (a["qe"] - b["qe"]) / a["qe"] < 0.10:
                pick = a
                reason = "smallest balanced configuration after which adding a neuron improves QE by < 10%"
                break
        print("\nQE per neuron count: " + "  ".join(f"{r['n']}: {r['qe']:.4f}" for r in res))
        print(f"Well-balanced configurations (no empty, none imbalanced): "
              f"{[r['n'] for r in valid] or 'none'}")
        print(f"\nMost suitable number of neurons: {pick['n']}  (QE {pick['qe']:.4f}; {reason})")

    # --- 4 -----------------------------------------------------------------
    def bmu_of(self, row_norm):
        return int(np.argmin(self.som.distances(row_norm)))

    def m4(self):
        hr("4. CUSTOMER CLUSTER MIGRATION ANALYZER")
        som = self.ensure()
        j = self.find_customer()
        f = ask("Feature to modify (1 = Annual Income, 2 = Spending Score)", 2, int, 1, 2)
        fi = f
        name = FEATS[fi]
        base_raw = self.raw[j].copy()
        base = self.bmu_of(self.norm(base_raw))
        print(f"\nCustomer {self.ids[j]}: Age {base_raw[0]:.0f}, Income {base_raw[1]:.0f}k$, "
              f"Spending {base_raw[2]:.0f} -> currently Cluster {base+1}")
        lo_v, hi_v = float(self.lo[fi]), float(self.hi[fi])

        def cluster_at(v):
            r = base_raw.copy()
            r[fi] = v
            return self.bmu_of(self.norm(r))

        def refine(a, b):                       # bisection on [a,b] where the BMU changes
            ca = cluster_at(a)
            for _ in range(30):
                m = (a + b) / 2
                if cluster_at(m) == ca:
                    a = m
                else:
                    b = m
            return b

        for label, step in (("INCREASING", +1.0), ("DECREASING", -1.0)):
            print(f"\n--- {name} {label} (1-unit steps, every 5th step shown + every migration) ---")
            v, cur, k = base_raw[fi], base, 0
            migrations = []
            while lo_v <= v + step <= hi_v:
                prev_v, v = v, v + step
                c = cluster_at(v)
                k += 1
                if c != cur:
                    exact = refine(prev_v, v)
                    migrations.append((exact, cur, c))
                    print(f"  {name} = {v:>6.1f} -> BMU neuron {c+1} (Cluster {c+1})   "
                          f"*** MIGRATION Cluster {cur+1} -> {c+1} at {name} = {exact:.2f} ***")
                    cur = c
                elif k % 5 == 0:
                    print(f"  {name} = {v:>6.1f} -> BMU neuron {c+1} (Cluster {c+1})")
            if migrations:
                for ex, a, b in migrations:
                    print(f"  => Customer migrates from Cluster {a+1} to Cluster {b+1} "
                          f"when {name} {'reaches' if step > 0 else 'falls to'} {ex:.2f}")
            else:
                print(f"  => No migration: customer stays in Cluster {base+1} across the whole range.")

    # --- 5 -----------------------------------------------------------------
    def m5(self):
        hr("5. CLUSTER PROTOTYPE GENERATOR")
        som = self.ensure()
        sizes, _ = self.cluster_sizes(som)
        print(f"Descriptions use dataset terciles (Age {self.p33[0]:.0f}/{self.p67[0]:.0f}, "
              f"Income {self.p33[1]:.0f}/{self.p67[1]:.0f}k$, Spending {self.p33[2]:.0f}/{self.p67[2]:.0f}) "
              f"applied to each learned prototype.\n")
        print(f"{'Cluster':>7} {'Size':>5} {'Age':>7} {'Income(k$)':>11} {'Spending':>9}")
        for k in range(som.n):
            p = self.denorm(som.W[k])
            print(f"{k+1:>7} {sizes[k]:>5} {p[0]:>7.1f} {p[1]:>11.1f} {p[2]:>9.1f}")
        print()
        for k in range(som.n):
            p = self.denorm(som.W[k])
            extra = "   (empty cluster)" if sizes[k] == 0 else ""
            print(f"Cluster {k+1}: {self.describe(p)}{extra}")

    # --- 6 -----------------------------------------------------------------
    def m6(self):
        hr("6. OUTLIER CUSTOMER DETECTOR")
        som = self.ensure()
        idx, dist = som.bmu(self.X)
        thr = dist.mean() + 2 * dist.std()
        order = np.argsort(-dist)
        print(f"BMU distance (normalised space): mean {dist.mean():.4f}, std {dist.std():.4f}; "
              f"outlier threshold = mean + 2*std = {thr:.4f}\n")
        flagged = [j for j in order if dist[j] > thr]
        print(f"Customers above threshold (potential outliers): {len(flagged)}")
        if flagged:
            print(f"{'CustID':>6} {'Age':>5} {'Income':>7} {'Spend':>6} {'Cluster':>8} {'BMU dist':>9}")
            for j in flagged:
                print(f"{self.ids[j]:>6} {self.raw[j,0]:>5.0f} {self.raw[j,1]:>7.0f} {self.raw[j,2]:>6.0f} "
                      f"{idx[j]+1:>8} {dist[j]:>9.4f}")
        print("\nTop 5 most unusual customers:")
        print(f"{'Rank':>4} {'CustID':>6} {'Age':>5} {'Income':>7} {'Spend':>6} {'Cluster':>8} {'BMU dist':>9}")
        for r, j in enumerate(order[:5], 1):
            print(f"{r:>4} {self.ids[j]:>6} {self.raw[j,0]:>5.0f} {self.raw[j,1]:>7.0f} {self.raw[j,2]:>6.0f} "
                  f"{idx[j]+1:>8} {dist[j]:>9.4f}")

    # --- 7 -----------------------------------------------------------------
    def m7(self):
        hr("7. SOM CLUSTER QUALITY COMPARISON")
        iters = ask("Training iterations", DEF_ITERS, int, 100, 100000)
        rows = []
        for model, comp in (("SOM", False), ("CLN", True)):
            for n in (2, 3, 4, 5, 6):
                for lr in (0.1, 0.3, 0.5):
                    som = self.train(n, lr, iters, comp)
                    sizes, _ = self.cluster_sizes(som)
                    rows.append(dict(model=model, n=n, lr=lr, it=iters,
                                     empty=int((sizes == 0).sum()), qe=som.quantization_error(self.X),
                                     big=int(sizes.max()), small=int(sizes.min())))
        print(f"\n{'Model':<5} {'Neurons':>7} {'LR':>5} {'Iters':>6} {'Empty':>6} {'QE':>8} "
              f"{'Largest':>8} {'Smallest':>9}")
        print("-" * 62)
        for r in rows:
            print(f"{r['model']:<5} {r['n']:>7} {r['lr']:>5} {r['it']:>6} {r['empty']:>6} {r['qe']:>8.4f} "
                  f"{r['big']:>8} {r['small']:>9}")
        print("-" * 62)
        ok = [r for r in rows if r["empty"] == 0]
        for n in (3, 4, 5):
            c = [r for r in ok if r["n"] == n]
            if c:
                b = min(c, key=lambda r: r["qe"])
                print(f"Best {n}-neuron configuration (no empty clusters): {b['model']} lr={b['lr']} "
                      f"QE={b['qe']:.4f}, sizes {b['small']}..{b['big']}")
        for m in ("SOM", "CLN"):
            q = [r["qe"] for r in rows if r["model"] == m]
            print(f"Average QE {m}: {np.mean(q):.4f}   empty clusters in total: "
                  f"{sum(r['empty'] for r in rows if r['model']==m)}")
        print("Note: QE always falls as neurons are added, so compare QE only together with "
              "empty/imbalanced clusters (see option 3).")


def main():
    s = System(sys.argv[1] if len(sys.argv) > 1 else None)
    actions = {"1": s.m1, "2": s.m2, "3": s.m3, "4": s.m4, "5": s.m5, "6": s.m6, "7": s.m7}
    while True:
        print("\n" + "=" * 78)
        print(" SOM CUSTOMER SEGMENTATION & CLUSTER EXPLORATION SYSTEM")
        print("=" * 78)
        for n, t in enumerate(["Customer Segmentation using SOM", "Best Matching Unit Explorer",
                               "Dynamic Cluster Count Challenge", "Customer Cluster Migration Analyzer",
                               "Cluster Prototype Generator", "Outlier Customer Detector",
                               "SOM Cluster Quality Comparison", "Exit"], 1):
            print(f" {n}. {t}")
        try:
            ch = input("Enter your choice (1-8): ").strip()
        except EOFError:
            break
        if ch == "8":
            print("Goodbye!")
            break
        if ch in actions:
            actions[ch]()
        else:
            print("Invalid choice. Please enter a number from 1 to 8.")


if __name__ == "__main__":
    main()



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
! WARNING: Mall_Customers.csv not found -> using SYNTHETIC DEMO DATA (200 customers)
! Put Mall_Customers.csv next to this script (or pass its path) for the real data.
! Results below are NOT real mall-customer results.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
[data] source = SYNTHETIC | 200 customers | features: Age, Annual Income (k$), Spending Score (min-max normalised to [0,1])

 SOM CUSTOMER SEGMENTATION & CLUSTER EXPLORATION SYSTEM
 1. Customer Segmentation using SOM
 2. Best Matching Unit Explorer
 3. Dynamic Cluster Count Challenge
 4. Customer Cluster Migration Analyzer
 5. Cluster Prototype Generator
 6. Outlier Customer Detector
 7. SOM Cluster Quality Comparison
 8. Exit
Enter your choice (1-8): 1

1. CUSTOMER SEGMENTATION USING SOM
Number of SOM neurons [4]: 2
Initial learning rate [0.5]: 3
Training iterations [3000]: 4

Trained SOM: 2 neurons, lr = 1.0, it